In [1]:
from leorbit.api import get_satellite, get_passes, Timestamp, Quantity, GPS, VisibleFromEarthLocationEvent, TimeInterval

Let's compute the position of a LEO satellite! Let's choose the ISS for the demonstration. 
Its NORAD Catalog ID is 25544.

If you have no idea of what a NORAD catalog ID is, you should definitely take 5min to check [Celestrak.org](https://celestrak.org)

In [2]:
iss = get_satellite(25544)
now = Timestamp.now()
c = iss.coordinates(now)
c.gps()

Requesting GP data for object n°25544 on `celestrak.com...`
GP data successfully fetched and stored locally.


<GPS:  007° 25′ 03″O,  045° 05′ 54″S>

Now that we have computed its position for the given time, we can project
it into any frame we want!

In [3]:
c.gcrf().human_repr("km")

'<Vector3 x=-3501.2988855192557 y=3283.5075674972536 z=-4816.584738610198 [km]>'

In [4]:
c.itrf().human_repr("km")

'<Vector3 x=4759.883615296329 y=-619.696538947629 z=-4816.584738610198 [km]>'

Horizontal coordinates in any Earth local frame!
For example, let's try in Paris.

In [5]:
gps_paris = GPS(
    longitude=2.333333 * Quantity.degree, 
    latitude=48.866667 * Quantity.degree, 
    altitude=0 * Quantity.meter
)
c.horizontal(gps_paris.earth_local_frame)

<Horizontal: Azimuth:  186° 53′ 10″, Altitude: - 045° 25′ 32″>

Ugh, negative altitude, it means it is not visible currently...

Well, I'd like to know next time it is visible in my area. Let's check when that happens, the next 7 days!

In [9]:
timeline = TimeInterval(
    start=now,
    stop=now + 7 * Quantity.day,
    dt=5 * Quantity.second
)
first_pass, *others = get_passes(iss, timeline, gps_paris, altitude_angle_min_degrees=0.)
first_pass

<TimeInterval from: '2026-04-28 at 00:19:17' to: '2026-04-28 at 00:28:52' dt: 5s>

Great! Now let's export the trajectory for the next pass:

In [10]:
iss.trajectory(first_pass).horizontal(gps_paris.earth_local_frame).to_csv(first_pass)

'timestamp,azimuth,elevation,range\n2026-04-28T00:19:17.926128+00:00,-159.4,0.1,6790.4\n2026-04-28T00:19:22.926128+00:00,-159.9,0.4,6790.4\n2026-04-28T00:19:27.926128+00:00,-160.3,0.6,6790.4\n2026-04-28T00:19:32.926128+00:00,-160.7,0.9,6790.4\n2026-04-28T00:19:37.926128+00:00,-161.2,1.2,6790.3\n2026-04-28T00:19:42.926128+00:00,-161.7,1.5,6790.3\n2026-04-28T00:19:47.926128+00:00,-162.2,1.8,6790.3\n2026-04-28T00:19:52.926128+00:00,-162.7,2.1,6790.3\n2026-04-28T00:19:57.926128+00:00,-163.2,2.4,6790.3\n2026-04-28T00:20:02.926128+00:00,-163.7,2.7,6790.3\n2026-04-28T00:20:07.926128+00:00,-164.3,3.0,6790.2\n2026-04-28T00:20:12.926128+00:00,-164.9,3.3,6790.2\n2026-04-28T00:20:17.926128+00:00,-165.4,3.6,6790.2\n2026-04-28T00:20:22.926128+00:00,-166.0,3.9,6790.2\n2026-04-28T00:20:27.926128+00:00,-166.7,4.2,6790.2\n2026-04-28T00:20:32.926128+00:00,-167.3,4.6,6790.2\n2026-04-28T00:20:37.926128+00:00,-168.0,4.9,6790.1\n2026-04-28T00:20:42.926128+00:00,-168.6,5.2,6790.1\n2026-04-28T00:20:47.926128+0